# 02 Graph Construction
## From Tabular Data to Graph Representations

| Dataset | Graph | Nodes | Edges |
|---------|-------|-------|-------|
| Credit Card | K-NN Similarity | Transactions | Cosine similarity |
| Medicare | Bipartite Projection | Providers | Shared drug patterns |

**Medicare pipeline this notebook:**
```
3x CMS CSVs  +  LEIE UPDATED.csv
     |                |
     v                v
 concat (3yr)    excluded NPIs
         \          /
          join on NPI --> fraud_label
               |
          build B(P,D)  [bipartite: providers <-> drugs]
               |
          project --> G(P,P)  [provider-provider, edge = shared drugs]
```


In [1]:
# ============================================================
# Google Colab Setup -- Run this cell FIRST every session
# ============================================================
import os, sys, glob

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ── Roots ────────────────────────────────────────────────────
DRIVE_ROOT   = "/content/drive/MyDrive/fraudDataset"
PROJECT_ROOT = f"{DRIVE_ROOT}/Fraud-Detection-GT"

# Add project src/ to Python path so we can import from src/
sys.path.insert(0, PROJECT_ROOT)

# ── Dataset Paths ─────────────────────────────────────────────
CC_PATH  = f"{DRIVE_ROOT}/creditcard/creditcard.csv"
LEIE_PATH = f"{DRIVE_ROOT}/leie/UPDATED.csv"

# Medicare: 3 separate yearly files (auto-detect names)
MED_DIR = f"{DRIVE_ROOT}/medicare"
MED_FILES = sorted(glob.glob(f"{MED_DIR}/*.csv"))  # finds all CSVs

# ── Output Paths (on Drive  →  survive session restarts) ─────
OUTPUTS_DIR = f"{PROJECT_ROOT}/outputs"
FIGURES_DIR = f"{PROJECT_ROOT}/outputs/figures"
METRICS_DIR = f"{PROJECT_ROOT}/outputs/metrics"
MODELS_DIR  = f"{PROJECT_ROOT}/outputs/models"
GRAPHS_DIR  = f"{PROJECT_ROOT}/data/graphs"

# ML baseline pickles (optional -- upload Fraud-Detection project too)
ML_PICKLES_DIR = f"{DRIVE_ROOT}/Fraud-Detection/pickled_storage"

# Create output directories (idempotent)
for _d in [OUTPUTS_DIR, FIGURES_DIR, METRICS_DIR, MODELS_DIR, GRAPHS_DIR,
           f"{GRAPHS_DIR}/credit_card", f"{GRAPHS_DIR}/medicare"]:
    os.makedirs(_d, exist_ok=True)

# Load config.yaml
import yaml
_cfg_path = f"{PROJECT_ROOT}/config.yaml"
CONFIG = {}
if os.path.exists(_cfg_path):
    with open(_cfg_path) as _f:
        CONFIG = yaml.safe_load(_f)

# ── Status Report ─────────────────────────────────────────────
print("=" * 58)
print("  Fraud Detection GT  |  Colab + Drive Environment")
print("=" * 58)
print(f"  Project Root  : {PROJECT_ROOT}")
print(f"  Credit Card   : {CC_PATH}")
print(f"  LEIE file     : {LEIE_PATH}")
print(f"  Medicare files: {len(MED_FILES)} found in {MED_DIR}")
for _f in MED_FILES:
    print(f"    - {os.path.basename(_f)}")
print()
print(f"  Outputs       : {OUTPUTS_DIR}")
print()

# Verify
_ok = True
for _path, _label in [(CC_PATH, "creditcard/creditcard.csv"),
                       (LEIE_PATH, "leie/UPDATED.csv")]:
    if os.path.exists(_path):
        _mb = os.path.getsize(_path)/1e6
        print(f"  [OK]  {_label} ({_mb:.0f} MB)")
    else:
        print(f"  [!!]  {_label} NOT FOUND")
        _ok = False

if len(MED_FILES) == 0:
    print("  [!!]  No Medicare CSVs found in medicare/")
    _ok = False
elif len(MED_FILES) < 3:
    print(f"  [??]  Only {len(MED_FILES)} Medicare file(s) found (expected 3)")
else:
    total_mb = sum(os.path.getsize(f)/1e6 for f in MED_FILES)
    print(f"  [OK]  medicare/ ({len(MED_FILES)} files, {total_mb:.0f} MB total)")

print()
print("  Ready!" if _ok else "  WARNING: Some files missing -- check Drive paths above")
print("=" * 58)


ValueError: mount failed

In [ ]:
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt, matplotlib, warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams.update({
    'figure.facecolor':'#0F0F1A', 'axes.facecolor':'#1A1A2E',
    'text.color':'#EEEEFF', 'axes.labelcolor':'#CCCCEE',
    'xtick.color':'#CCCCEE', 'ytick.color':'#CCCCEE',
})
from src.graph_builder import (CreditCardGraphBuilder, MedicareGraphBuilder,
                                 save_graph, graph_summary)
print("Setup complete")


---
## 1. Credit Card: K-NN Similarity Graph

Every transaction = node.
Edges connect the k most similar transactions in PCA feature space (cosine similarity).


In [ ]:
cc_builder = CreditCardGraphBuilder(CONFIG)
G_cc = cc_builder.build(CC_PATH)

s = graph_summary(G_cc)
print("\nCredit Card Graph:")
for k,v in s.items(): print(f"  {k:20s}: {v}")

save_graph(G_cc, f'{GRAPHS_DIR}/credit_card/G_cc_knn.pkl')


In [ ]:

# Fraud ego-subgraph visualization
import random; random.seed(42)
fraud_nodes = [n for n in G_cc.nodes() if G_cc.nodes[n].get('label')==1]
sample_fn   = random.sample(fraud_nodes, min(25, len(fraud_nodes)))
sub_nodes   = set(sample_fn)
for fn in sample_fn: sub_nodes.update(list(G_cc.neighbors(fn))[:4])
G_sub = G_cc.subgraph(list(sub_nodes))

fig, ax = plt.subplots(figsize=(10,8))
ax.set_facecolor('#0F0F1A'); fig.patch.set_facecolor('#0F0F1A')
pos = nx.spring_layout(G_sub, seed=42, k=0.6)
nc  = ['#FF4444' if G_cc.nodes[n].get('label')==1 else '#4488FF' for n in G_sub.nodes()]
ns  = [250 if G_cc.nodes[n].get('label')==1 else 40 for n in G_sub.nodes()]
nx.draw_networkx_nodes(G_sub,pos,node_color=nc,node_size=ns,alpha=0.9,ax=ax)
nx.draw_networkx_edges(G_sub,pos,alpha=0.25,edge_color='#8888BB',width=0.5,ax=ax)

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0],[0],marker='o',color='w',markerfacecolor='#FF4444',markersize=10,label='Fraud'),
    Line2D([0],[0],marker='o',color='w',markerfacecolor='#4488FF',markersize=8, label='Normal'),
], loc='upper right', facecolor='#1A1A2E', edgecolor='#555')
ax.set_title('Credit Card: Fraud Ego-Subgraph', color='#EEEEFF', fontsize=12)
ax.axis('off'); plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/cc_fraud_subgraph.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 2. Medicare: Provider-Drug Bipartite -> Provider-Provider Projection

**Data sources:**
- `medicare/` folder: 3 CSV files (2017, 2018, 2019) auto-loaded
- `leie/UPDATED.csv`: OIG exclusion list joined on NPI

**Memory note:** Full 3-year Medicare is ~4.5 GB.
The builder subsamples to `max_providers` (config: 30,000) automatically.


In [ ]:
# ============================================================
# Medicare Graph Construction — Memory-Safe Version
# ============================================================
#
# WHY THE ORIGINAL CRASHED (176 GB RAM):
#
#   Problem 1 — 75M rows loaded at once:
#     3 files × 25M rows = 75 million rows → ~30 GB RAM before any processing
#     Fix: Sample 2M rows per file → 6M total, still statistically representative
#
#   Problem 2 — Hub drug explosion (density = 0.74):
#     Drugs like metformin, lisinopril are prescribed by >50% of all providers.
#     When projected, EVERY provider shares these drugs → near-complete graph.
#     334 MILLION edges × ~100 bytes = 33+ GB just for edge storage.
#     Fix: Remove drugs prescribed by >25% of providers before projection.
#     Only keep "specialty signal" drugs that distinguish provider behavior.
#
#   Problem 3 — Full O(n²) projection:
#     networkx weighted_projected_graph creates ALL possible pairs.
#     Fix: Manual projection with minimum shared drug threshold (>=5) + top-K.
#
# EXPECTED RESULT AFTER FIX:
#   Edges: ~100K-500K (vs 334M), Density: ~0.001-0.01 (vs 0.74)
# ============================================================

import gc
from collections import defaultdict

# ── Configuration ────────────────────────────────────────────
ROWS_PER_FILE    = 2_000_000  # Rows sampled per yearly CSV (2M × 3 = 6M total)
MAX_PROVIDERS    = 8_000      # Max providers in final graph
MAX_DRUG_PREV    = 0.25       # Remove drugs prescribed by >25% of providers (hub drugs)
MIN_DRUG_PREV    = 0.005      # Remove drugs prescribed by <0.5% of providers (too rare)
MIN_SHARED_DRUGS = 5          # Minimum shared specialty drugs to create an edge
TOP_K_NEIGHBORS  = 25         # Keep only top-25 strongest edges per provider
# ─────────────────────────────────────────────────────────────

# Step 1: Load Medicare — sampled to avoid memory crash
print("Step 1: Loading Medicare files (sampled)...")
med_files = sorted(glob.glob(f"{MED_DIR}/*.csv"))
dfs = []
for fp in med_files:
    yr = next((y for y in ['2017','2018','2019'] if y in fp), None)
    df_tmp = pd.read_csv(fp, nrows=ROWS_PER_FILE, low_memory=False)
    if yr: df_tmp['year'] = int(yr)
    dfs.append(df_tmp)
    print(f"  {os.path.basename(fp)}: {len(df_tmp):,} rows loaded")

df_med = pd.concat(dfs, ignore_index=True)
del dfs; gc.collect()
print(f"  Combined: {len(df_med):,} rows | {df_med.shape[1]} cols")

# Step 2: Detect column names (robust to CMS naming variations across years)
print("\nStep 2: Detecting column names...")
def _find_col(df, candidates):
    cl = {c.lower(): c for c in df.columns}
    return next((cl[c.lower()] for c in candidates if c.lower() in cl), None)

NPI_C   = _find_col(df_med, ['Prscrbr_NPI', 'npi', 'NPI'])
DRUG_C  = _find_col(df_med, ['Gnrc_Name', 'gnrc_name', 'Brnd_Name'])
CLM_C   = _find_col(df_med, ['Tot_Clms', 'tot_clms'])
STATE_C = _find_col(df_med, ['Prscrbr_State_Abrvtn', 'prscrbr_state_abrvtn'])
print(f"  NPI={NPI_C} | Drug={DRUG_C} | Claims={CLM_C} | State={STATE_C}")

# Step 3: Load LEIE — extract valid 10-digit NPIs
print("\nStep 3: Loading LEIE exclusion list...")
leie_df   = pd.read_csv(LEIE_PATH, low_memory=False)
leie_npis = set(
    str(n).strip() for n in leie_df['NPI'].dropna()
    if str(n).strip().isdigit() and len(str(n).strip()) == 10
)
print(f"  Valid excluded NPIs: {len(leie_npis):,}")

# Step 4: Clean and label
print("\nStep 4: Cleaning data and assigning fraud labels...")
df_c = df_med[[NPI_C, DRUG_C, CLM_C, STATE_C]].dropna(subset=[NPI_C, DRUG_C]).copy()
df_c[NPI_C]        = df_c[NPI_C].astype(str).str.strip()
df_c[DRUG_C]       = df_c[DRUG_C].astype(str).str.strip().str.upper()
df_c['fraud_label'] = df_c[NPI_C].isin(leie_npis).astype(int)
del df_med; gc.collect()

# Step 5: Stratified provider subsample (keep ALL fraud providers)
print("\nStep 5: Stratified provider sampling...")
prov_labels = df_c.groupby(NPI_C)['fraud_label'].max()
fraud_prov  = prov_labels[prov_labels == 1].index.tolist()
normal_prov = prov_labels[prov_labels == 0].index.tolist()
n_normal    = min(len(normal_prov), MAX_PROVIDERS - len(fraud_prov))
np.random.seed(42)
keep_prov   = set(fraud_prov) | set(np.random.choice(normal_prov, n_normal, replace=False))
df_c        = df_c[df_c[NPI_C].isin(keep_prov)].copy()
print(f"  Providers: {len(keep_prov):,} total | {len(fraud_prov):,} fraud | {n_normal:,} normal")

# Step 6: Remove hub drugs — THE CRITICAL FIX
# Drugs prescribed by >25% of providers create near-complete graphs.
# We keep only "discriminative" drugs that signal unusual prescribing patterns.
print("\nStep 6: Removing hub drugs (the key fix for density explosion)...")
n_prov    = df_c[NPI_C].nunique()
drug_prev = df_c.groupby(DRUG_C)[NPI_C].nunique() / n_prov

valid_drugs = drug_prev[(drug_prev <= MAX_DRUG_PREV) & (drug_prev >= MIN_DRUG_PREV)].index
n_hub  = (drug_prev > MAX_DRUG_PREV).sum()
n_rare = (drug_prev < MIN_DRUG_PREV).sum()
df_c   = df_c[df_c[DRUG_C].isin(valid_drugs)].copy()
print(f"  Drugs kept   : {len(valid_drugs):,}")
print(f"  Hub removed  : {n_hub:,}  (e.g. metformin, lisinopril — too common)")
print(f"  Rare removed : {n_rare:,}  (prescribed by <0.5% of providers)")

# Step 7: Build bipartite graph
print("\nStep 7: Building bipartite Provider-Drug graph...")
B_med = nx.Graph()
unique_prov  = df_c[NPI_C].unique()
unique_drugs = df_c[DRUG_C].unique()

for p in unique_prov:
    B_med.add_node(f"P_{p}", bipartite=0, node_type='provider',
                   label=int(prov_labels.get(p, 0)),
                   tot_clms=float(df_c[df_c[NPI_C]==p][CLM_C].sum()) if CLM_C else 0.0)
for d in unique_drugs:
    B_med.add_node(f"D_{d}", bipartite=1, node_type='drug', label=0)

edge_pairs = df_c[[NPI_C, DRUG_C]].drop_duplicates().values
for prov, drug in edge_pairs:
    B_med.add_edge(f"P_{prov}", f"D_{drug}")

fraud_b = sum(1 for n,d in B_med.nodes(data=True) if d.get('label')==1)
print(f"  Bipartite: {B_med.number_of_nodes():,} nodes | {B_med.number_of_edges():,} edges | {fraud_b} fraud providers")

# Step 8: Manual projection with min-weight + top-K filters
# Instead of networkx's full O(n²) projection, we:
#   (a) Count shared specialty drugs between each pair of providers
#   (b) Only create edge if shared count >= MIN_SHARED_DRUGS
#   (c) Keep only top TOP_K_NEIGHBORS strongest connections per provider
print(f"\nStep 8: Projecting to Provider-Provider graph...")
print(f"  min_shared={MIN_SHARED_DRUGS}, top_k={TOP_K_NEIGHBORS} — this prevents edge explosion")

prov_set      = {n for n,d in B_med.nodes(data=True) if d.get('bipartite')==0}
drug_to_provs = defaultdict(list)
for prov in prov_set:
    for drug in B_med.neighbors(prov):
        drug_to_provs[drug].append(prov)

pair_weights = defaultdict(int)
for drug, provs in drug_to_provs.items():
    for i in range(len(provs)):
        for j in range(i+1, len(provs)):
            pair_weights[(provs[i], provs[j])] += 1

del drug_to_provs; gc.collect()

# Build node-level candidate lists (filtered by min weight)
node_candidates = defaultdict(list)
for (p1, p2), w in pair_weights.items():
    if w >= MIN_SHARED_DRUGS:
        node_candidates[p1].append((w, p2))
        node_candidates[p2].append((w, p1))

del pair_weights; gc.collect()

G_med = nx.Graph()
for n in prov_set:
    G_med.add_node(n, **B_med.nodes[n])

for node, neighbors in node_candidates.items():
    for w, nb in sorted(neighbors, reverse=True)[:TOP_K_NEIGHBORS]:
        if not G_med.has_edge(node, nb):
            G_med.add_edge(node, nb, weight=w)

del node_candidates; gc.collect()

# Results
s_med   = graph_summary(G_med)
col_info = {'npi': NPI_C, 'drug': DRUG_C, 'label': 'fraud_label'}
df_clean = df_c

print("\nMedicare Provider Graph:")
for k, v in s_med.items():
    print(f"  {k:20s}: {v}")

save_graph(G_med, f'{GRAPHS_DIR}/medicare/G_med_provider.pkl')
save_graph(B_med, f'{GRAPHS_DIR}/medicare/B_med_bipartite.pkl')
print(f"\nSaved both graphs to {GRAPHS_DIR}/medicare/")


In [ ]:
# Stats and plots
prov_nodes = [n for n,d in B_med.nodes(data=True) if d.get('bipartite')==0]
drug_nodes = [n for n,d in B_med.nodes(data=True) if d.get('bipartite')==1]
fraud_prov = [n for n in prov_nodes if B_med.nodes[n].get('label')==1]
print(f"Bipartite graph:")
print(f"  Providers : {len(prov_nodes):,}")
print(f"  Drugs     : {len(drug_nodes):,}")
print(f"  Edges     : {B_med.number_of_edges():,}")
print(f"  Fraud prov: {len(fraud_prov):,} ({len(fraud_prov)/max(len(prov_nodes),1)*100:.2f}%)")

# Degree comparison
frd_deg = [G_med.degree(n) for n in G_med.nodes() if G_med.nodes[n].get('label')==1]
nrm_deg = [G_med.degree(n) for n in G_med.nodes() if G_med.nodes[n].get('label')==0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Medicare Provider Graph Analysis', fontsize=13, color='#EEEEFF', fontweight='bold')

ax = axes[0]
if frd_deg:
    ax.hist(nrm_deg, bins=50, alpha=0.6, color='#4488FF',
            label=f'Normal ({len(nrm_deg):,})', density=True)
    ax.hist(frd_deg, bins=50, alpha=0.8, color='#FF4444',
            label=f'Fraud  ({len(frd_deg):,})', density=True)
    print(f"Avg degree  Fraud: {np.mean(frd_deg):.1f}  Normal: {np.mean(nrm_deg):.1f}")
else:
    ax.text(0.5,0.5,'No fraud labels in graph sample',
            ha='center',va='center',transform=ax.transAxes,color='#EEEEFF')
ax.set_title('Provider Degree by Fraud Label'); ax.set_xlabel('Degree')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
weights=[d.get('weight',1) for _,_,d in G_med.edges(data=True)]
ax.hist(weights, bins=50, color='#7788FF', alpha=0.85, edgecolor='#333')
ax.set_xlabel('Edge Weight (shared drugs)'); ax.set_ylabel('Count')
ax.set_title('Edge Weight Distribution'); ax.set_yscale('log'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/medicare_graph_construction.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
print("=" * 58)
print("  NOTEBOOK 02 COMPLETE")
print("=" * 58)
print(f"  CC  Graph : {s['n_nodes']:,} nodes | {s['n_edges']:,} edges | fraud={s['fraud_rate']*100:.2f}%")
print(f"  Med Graph : {s_med['n_nodes']:,} nodes | {s_med['n_edges']:,} edges | fraud={s_med['fraud_rate']*100:.2f}%")
print("  Graphs saved to Drive")
print("  Next -> 03_Classical_Graph_Analysis.ipynb")
print("=" * 58)
